# DB7-021: task-aware, bounded BRB weight corrections

This notebook tests whether optimizing the **final 17-class gesture prediction** makes BRB fusion more useful. It reuses the verified DB7-018 predictions; it does not train any neural network or change signal preprocessing.

## Prespecified comparisons
1. **Frozen hybrid:** the DB7-020 waveform/spectral BRBs plus calibrated inertial confidence.
2. **Task-aware BRB:** two new eight-rule BRBs make bounded adjustments to the waveform and spectral weights. They optimize final gesture cross-entropy.
3. **Task-aware logistic:** two logistic heads receive the same three antecedents and optimize the same final gesture loss under the same bounds. This checks whether rule inference adds value over a simpler model. BRB and logistic have different parameter counts (16 versus8); they have equal search budgets, not equal capacity.
4. **Correctness-loss BRB:** identical residual BRB architecture, but its raw head scores are trained against each EMG expert's binary correctness. This matched control separates the objective from the addition of a residual adjustment layer.

Original BRB, confidence weighting, global weighting, inertial-only and joint SI/WSI are reported as contextual controls. The SI/WSI predictions are not inputs to any gate and are never used for fitting.

## Fixed data protocol
All22 subjects; separate within-subject systems; frozen neural seed42. Exercise B is E1 restimulus labels1–17. Active gesture segments exclude rest. EMG12 channels receive zero-phase20–450Hz bandpass and50Hz notch; inertial ACC/gyro/mag each36 channels are aligned per segment. Training-only channel scaling retains offsets relative to the training mean; there is no per-window inertial centering.

The inherited outer window is200ms,400 samples at2kHz, with10ms stride. Each window contains three100ms frames at50ms internal hops. Final experts were trained for13 fixed epochs on repetitions1/3/4/6; test repetitions are2/5. No new neural seed is generated here.

## Bounded adjustment and exact inputs/outputs
Inputs per window: expert probabilities `p[N,3,17]`, baseline effective weights `w0[N,3]`, and indicators `q[N,3,3]` (normalized entropy, mean pairwise total-variation disagreement, and training-reference deviation).

Each new W/S head sees its own three indicators. Low/high references0/1 produce eight rules for BRB; analytical RIMER combines their binary conclusions. Each head score `a` is in[0,1]. Its consequent means **support for increasing versus decreasing an expert's fusion weight**, not a calibrated correctness probability. For the matched correctness-loss control only, the fitting target is expert correctness.

`delta_W = cap * (2*a_W - 1)`; likewise for S; `delta_I = 0`.

`w = softmax(log(w0) + delta)` and `P(c) = sum_j w_j p_j(c)`.

Outputs: adjusted weights[N,3], log-weight changes[N,3], final probabilities[N,17], and one gesture label per window. The inertial score and unnormalized log-weight stay fixed, although its normalized weight can change. The bound limits each EMG/inertial weight ratio by a factor exp(cap); it is not an absolute percentage-point weight bound. W/S relative odds can change by exp(2*cap).

All-zero head parameters give scores0.5 and reproduce the baseline exactly. Parameters are bounded to[-8,8]. L-BFGS-B uses analytical gradients. Failed/nonfinite/worse optimization is recorded; nonfinite or worse candidates fall back to neutral. Finite improved unconverged solutions are retained with an explicit warning status, never relabeled converged.

## Training objectives
Task-aware BRB and logistic minimize repetition-weighted final gesture cross-entropy, plus `lambda * mean(sum((w-w0)^2))` and `0.0001 * mean(theta^2)`.

The correctness-BRB replaces gesture cross-entropy with the mean binary cross-entropy of its two head scores against W/S expert correctness. It retains the same correction mapping and penalties. All target labels are training-side only.

## Development folds and locking
Three meta-development folds hold out repetition1,3 or4. The other two repetitions fit a freshly reconstructed reference meta model (temperatures, alpha and BRB heads). Temperature crossfitting occurs within those two fit repetitions. Repetition6 calibrates the reference W/S BRB outputs and inertial confidence. The original full meta model is not used to initialize these development baselines.

The new corrections fit on the two fit repetitions and are scored on the held repetition. Each of the22 subjects and each held repetition contributes equally. The common search budget per method is:
- Neutral cap0.
- cap=log(1.25) or log(2), crossed with lambda0.1 or1.

Select separately for each method: lowest mean held-repetition NLL among candidates whose mean held accuracy is at least the same-fold baseline. Tie-break smaller cap, then larger penalty. If no candidate improves NLL without reducing mean accuracy, choose neutral. This selection constraint does not guarantee improved test accuracy or protect every subject.

Save `SELECTION_LOCKED_BEFORE_TEST.json` before reading test arrays. Refit the selected corrections on OOF repetitions1/3/4 around the original frozen DB7-020 hybrid, with scalar confidence calibration fromOOF6. Evaluate test2/5 after fitting. There is no test-based selection or automatic retry with new settings.

**Limitation:** the saved OOF neural predictions and shift references share underlying neural training histories. These are exploratory meta-development folds, not independent nested validation. The baseline heads also see the adjustment fitting rows. The inspected test recordings and one frozen seed cannot support a new confirmatory superiority claim.

## Outputs and statistical comparisons
Save the full development grid, selected settings, optimizer statuses, fold-specific reference models, final initial/trained BRB rules, logistic coefficients, test weights/predictions, confusion matrices, errors per subject/gesture/repetition, corrected/harmed counts, accuracy, macro F1, balanced accuracy, NLL, multiclass Brier and ECE10.

Primary paired subject contrasts: taskBRB vs hybrid, tasklogistic vs hybrid, taskBRB vs tasklogistic, and taskBRB vs correctnessBRB. Use100,000 subject sign-flips with Holm correction across4 comparisons; subject bootstrap95% intervals use20,000 resamples and are not multiplicity-adjusted. Window McNemar is descriptive because95%-overlapping windows are correlated. The selected settings may differ across methods: comparisons assess equally tuned pipelines rather than the loss gradient alone at identical hyperparameters.

CPU-only workload:66 development folds and990 method/configuration evaluations (including198 neutral entries), followed by66 final method refits/evaluations. Zero neural fits. GPU budget is unused. The source3GB result archive SHA256 and original BRB replay are checked. A separate compact summary artifact avoids downloading all per-window traces.


In [ ]:
import os
os.environ['OPENBLAS_NUM_THREADS']='1'
os.environ['OMP_NUM_THREADS']='1'
from pathlib import Path
SOURCE=Path('/kaggle/working/db7_021_source')
SOURCE.mkdir(exist_ok=True)


## brb_meta.py
Frozen baseline BRB functions from DB7-018. These reconstruct development baselines and replay the unchanged final model.

In [ ]:
%%writefile /kaggle/working/db7_021_source/brb_meta.py
"""Training-only reliability fusion for DB7 W/S/I expert logits.

Labels are zero based. fit_meta accepts OOF predictions for repetitions 1/3/4/6
only. Repetition 6 is reserved for reliability calibration, not model fitting,
temperature selection or global fusion weights. The caller must supply shifts
computed against each OOF base model's own training-only signal references.

This is nested development, NOT independent meta cross-validation: OOF base
models may share training recordings. Outer test predictions never enter fit.
"""
from __future__ import annotations

import csv
import hashlib
import json
from pathlib import Path

import numpy as np
from scipy.optimize import minimize
from scipy.special import expit, logit, logsumexp, softmax

VERSION = "db7-brb-meta-v1"
EXPERTS = ("W", "S", "I")
FIT_REPS = (1, 3, 4)
CAL_REP = 6
EPS = 1e-9
HEAD_L2 = 0.01
CAL_L2 = 0.01
ALPHA_L2 = 0.005
MIN_EVENTS = 5
MAXITER = 180
RULE_BITS = np.array([[int(x) for x in f"{r:03b}"] for r in range(8)])


def _inputs(logits, shifts, y=None, repetitions=None):
    z = np.asarray(logits, dtype=np.float64)
    s = np.asarray(shifts, dtype=np.float64)
    if z.ndim != 3 or z.shape[1:] != (3, 17) or len(z) == 0:
        raise ValueError("logits must have nonempty shape [N,3,17]")
    if s.shape != z.shape[:2] or not np.isfinite(z).all() or not np.isfinite(s).all():
        raise ValueError("finite shifts [N,3] and logits required")
    if np.any((s < 0) | (s > 1)):
        raise ValueError("training-reference shifts must be in [0,1]")
    if y is None:
        return z, s
    yy = np.asarray(y)
    rr = np.asarray(repetitions)
    if yy.shape != (len(z),) or rr.shape != yy.shape:
        raise ValueError("y and repetitions must have shape [N]")
    if not np.isin(yy, np.arange(17)).all():
        raise ValueError("y must contain zero-based labels 0..16")
    if not np.isin(rr, (*FIT_REPS, CAL_REP)).all():
        raise ValueError("fit_meta accepts only training repetitions 1,3,4,6; test forbidden")
    if set(rr.tolist()) != {1, 3, 4, 6}:
        raise ValueError("all meta-fit repetitions 1/3/4 and calibration repetition 6 required")
    return z, s, yy.astype(np.int64), rr.astype(np.int64)


def _group_weights(groups):
    """Equal repetition weight; do not treat differing window counts as trials."""
    groups = np.asarray(groups)
    levels, counts = np.unique(groups, return_counts=True)
    return np.array([1.0 / (len(levels) * counts[np.searchsorted(levels, g)]) for g in groups])


def _binary_loss(target, predicted, weights):
    p = np.clip(predicted, EPS, 1 - EPS)
    return float(-np.sum(weights * (target * np.log(p) + (1 - target) * np.log1p(-p))))


def _optimization(result):
    return {"success": bool(result.success), "message": str(result.message),
            "iterations": int(result.nit), "objective": float(result.fun)}


def fit_temperatures(logits, y, groups):
    """Positive scalar temperature per expert, using supplied development rows."""
    weights = _group_weights(groups)
    temperatures, diagnostics = [], []
    for j in range(3):
        z = logits[:, j]
        def objective(theta):
            zz = z / np.exp(theta[0])
            p = softmax(zz, axis=1)
            loss = np.sum(weights * (logsumexp(zz, axis=1) - zz[np.arange(len(y)), y]))
            grad = np.sum(weights * (zz[np.arange(len(y)), y] - np.sum(p * zz, axis=1)))
            return float(loss + 0.001 * theta[0] ** 2), np.array([grad + .002 * theta[0]])
        opt = minimize(objective, [0.], jac=True, method="L-BFGS-B",
                       bounds=[(np.log(.05), np.log(20.))], options={"maxiter": MAXITER})
        if not np.isfinite(opt.fun):
            raise RuntimeError("nonfinite temperature optimization")
        temperatures.append(float(np.exp(opt.x[0])))
        diagnostics.append(_optimization(opt))
    return temperatures, diagnostics


def _probabilities(logits, temperatures):
    return softmax(logits / np.asarray(temperatures)[None, :, None], axis=2)


def indicators(probabilities, shifts):
    """Expert entropy, mean pairwise TV and precomputed training deviation."""
    p = np.asarray(probabilities, dtype=float)
    entropy = -np.sum(p * np.log(np.clip(p, EPS, 1)), axis=2) / np.log(17.)
    disagreement = np.zeros(p.shape[:2])
    for j in range(3):
        disagreement[:, j] = sum(.5 * np.abs(p[:, j] - p[:, k]).sum(1)
                                for k in range(3) if k != j) / 2
    return np.clip(np.stack([entropy, disagreement, shifts], axis=-1), 0, 1)


def rule_activations(q):
    """Product reference matching: [...,3] -> [...,8], sum exactly one."""
    q = np.asarray(q, dtype=float)
    if q.shape[-1] != 3 or not np.isfinite(q).all() or np.any((q < 0) | (q > 1)):
        raise ValueError("rule indicators must be finite [...,3] in [0,1]")
    a = np.prod(np.where(RULE_BITS, q[..., None, :], 1 - q[..., None, :]), axis=-1)
    return a / a.sum(axis=-1, keepdims=True)


def rimer_correct(activations, correct_beliefs, return_jacobian=False):
    """Analytical ER/RIMER for complete binary rule conclusions.

    A_n=prod(1-w+w*beta_n), B=prod(1-w), beta_n=(A_n-B)/(A0+A1-2B).
    Normalized activation is rule evidence weight. Returned jacobian is with
    respect to each rule's correctness belief (NOT its logit).
    """
    w = np.asarray(activations, dtype=float)
    b = np.asarray(correct_beliefs, dtype=float)
    if w.shape[-1] != 8 or b.shape != (8,):
        raise ValueError("eight activations and eight correctness beliefs required")
    if not np.isfinite(w).all() or not np.isfinite(b).all() or np.any((b < 0) | (b > 1)):
        raise ValueError("invalid ER beliefs")
    if np.any((w < 0) | (w > 1)) or not np.allclose(w.sum(-1), 1):
        raise ValueError("ER activations must be normalized")
    f1 = 1 - w + w * b
    f0 = 1 - w + w * (1 - b)
    a1, a0, bb = f1.prod(-1), f0.prod(-1), (1 - w).prod(-1)
    denom = a1 + a0 - 2 * bb
    if np.any(denom <= 0):
        raise FloatingPointError("degenerate ER normalization")
    p = np.clip((a1 - bb) / denom, 0, 1)
    if not return_jacobian:
        return p
    # Product excluding each factor handles exact zero factors at rule vertices.
    da1 = np.stack([w[..., k] * np.delete(f1, k, axis=-1).prod(-1) for k in range(8)], -1)
    da0 = -np.stack([w[..., k] * np.delete(f0, k, axis=-1).prod(-1) for k in range(8)], -1)
    jac = (da1 * denom[..., None] - (a1 - bb)[..., None] * (da1 + da0)) / denom[..., None] ** 2
    return p, jac


def _fit_alpha(p, y, weights):
    true_p = p[np.arange(len(p))[:, None], np.arange(3)[None, :], y[:, None]]
    def objective(theta):
        alpha = softmax(theta)
        mixture = np.clip(true_p @ alpha, EPS, 1)
        loss = -np.sum(weights * np.log(mixture)) + ALPHA_L2 * np.sum(theta ** 2)
        da = -np.sum(weights[:, None] * true_p / mixture[:, None], axis=0)
        grad = alpha * (da - alpha @ da) + 2 * ALPHA_L2 * theta
        return float(loss), grad
    opt = minimize(objective, np.zeros(3), jac=True, method="L-BFGS-B",
                   bounds=[(-6, 6)] * 3, options={"maxiter": MAXITER})
    return softmax(opt.x).tolist(), _optimization(opt)


def _raw_head(head, q):
    if head["kind"] == "constant":
        return np.full(len(q), head["value"])
    if head["kind"] == "logistic":
        return expit(np.column_stack([np.ones(len(q)), q - .5]) @ np.asarray(head["parameters"]))
    qq = q.copy()
    if head["kind"] == "brb_no_shift":
        qq[:, 2] = .5
    a = rule_activations(qq)
    beliefs = expit(head["parameters"])
    return a @ beliefs if head["kind"] == "sugeno" else rimer_correct(a, beliefs)


def _fit_head(kind, q, target, weights):
    prior = float((np.sum(target) + .5) / (len(target) + 1))
    if min(int(target.sum()), int((1 - target).sum())) < MIN_EVENTS:
        return {"kind": "constant", "requested_kind": kind, "value": prior,
                "fallback": "fewer than five correct or incorrect examples"}
    if kind == "logistic":
        x = np.column_stack([np.ones(len(q)), q - .5])
        center = np.array([logit(prior), 0, 0, 0])
        def objective(theta):
            raw = expit(x @ theta)
            loss = _binary_loss(target, raw, weights) + HEAD_L2 * np.mean((theta - center) ** 2)
            grad = x.T @ (weights * (raw - target)) + 2 * HEAD_L2 * (theta - center) / len(theta)
            return loss, grad
    else:
        qq = q.copy()
        if kind == "brb_no_shift":
            qq[:, 2] = .5
        a = rule_activations(qq)
        center = np.full(8, logit(prior))
        def objective(theta):
            beliefs = expit(theta)
            if kind == "sugeno":
                raw, jac = a @ beliefs, a
            else:
                raw, jac = rimer_correct(a, beliefs, return_jacobian=True)
            pp = np.clip(raw, EPS, 1 - EPS)
            derivative = weights * (pp - target) / (pp * (1 - pp))
            grad = (derivative @ jac) * beliefs * (1 - beliefs)
            loss = _binary_loss(target, pp, weights) + HEAD_L2 * np.mean((theta - center) ** 2)
            grad += 2 * HEAD_L2 * (theta - center) / len(theta)
            return loss, grad
    opt = minimize(objective, center, jac=True, method="L-BFGS-B", bounds=[(-10, 10)] * len(center),
                   options={"maxiter": MAXITER, "ftol": 1e-9})
    if not np.isfinite(opt.fun) or not np.isfinite(opt.x).all():
        raise RuntimeError(f"nonfinite {kind} fit")
    return {"kind": kind, "parameters": opt.x.tolist(), "optimization": _optimization(opt), "prior": prior}


def _fit_calibration(raw, target):
    """Monotone logit-affine reliability calibration on repetition 6 only."""
    if len(target) < 20:
        return {"slope": 1., "intercept": 0., "fallback": "fewer than 20 calibration rows"}
    x = logit(np.clip(raw, 1e-5, 1 - 1e-5))
    sparse = min(int(target.sum()), int((1 - target).sum())) < MIN_EVENTS
    def objective(theta):
        slope, intercept = theta
        p = expit(slope * x + intercept)
        weights = np.full(len(target), 1 / len(target))
        loss = _binary_loss(target, p, weights) + CAL_L2 * ((slope - 1) ** 2 + intercept ** 2)
        residual = p - target
        grad = np.array([np.mean(residual * x) + 2 * CAL_L2 * (slope - 1),
                         np.mean(residual) + 2 * CAL_L2 * intercept])
        return loss, grad
    opt = minimize(objective, [1., 0.], jac=True, method="L-BFGS-B",
                   bounds=[(1., 1.) if sparse else (0., 5.), (-8., 8.)], options={"maxiter": MAXITER})
    return {"slope": float(opt.x[0]), "intercept": float(opt.x[1]), "optimization": _optimization(opt),
            "fallback": "intercept-only: fewer than five events in one outcome" if sparse else None}


def _calibrated(raw, calibration):
    return expit(calibration["slope"] * logit(np.clip(raw, 1e-5, 1 - 1e-5)) + calibration["intercept"])


def _weights(alpha, reliability):
    unnormalized = np.asarray(alpha)[None, :] * np.clip(reliability, 0, 1)
    denominator = unnormalized.sum(1, keepdims=True)
    return np.divide(unnormalized, denominator, out=np.broadcast_to(alpha, unnormalized.shape).copy(), where=denominator > EPS)


def predict_diagnostics(model, logits, shifts):
    z, s = _inputs(logits, shifts)
    if model.get("version") != VERSION:
        raise ValueError("unsupported meta model version")
    p = _probabilities(z, model["temperatures"])
    q = indicators(p, s)
    reliabilities = {"confidence_weight": p.max(2)}
    raw_reliabilities = {}
    for name, heads in model["heads"].items():
        raw = np.column_stack([_raw_head(heads[j], q[:, j]) for j in range(3)])
        raw_reliabilities[name] = raw
        reliabilities[name] = np.column_stack([_calibrated(raw[:, j], model["reliability_calibrations"][name][j]) for j in range(3)])
    weights = {name: _weights(model["alpha"], r) for name, r in reliabilities.items()}
    return {"expert_probabilities": p, "indicators": q, "raw_reliabilities": raw_reliabilities,
            "reliabilities": reliabilities, "weights": weights, "rule_activations": rule_activations(q)}


def predict_meta(model, logits, shifts):
    d = predict_diagnostics(model, logits, shifts)
    p = d["expert_probabilities"]
    result = {f"expert_{name.lower()}": p[:, j] for j, name in enumerate(EXPERTS)}
    result["mean"] = p.mean(1)
    result["global_weight"] = np.sum(p * np.asarray(model["alpha"])[None, :, None], axis=1)
    result.update({name: np.sum(p * w[:, :, None], axis=1) for name, w in d["weights"].items()})
    return result


def _metrics(y, p):
    confidence, predicted = p.max(1), p.argmax(1)
    correct = predicted == y
    onehot = np.eye(p.shape[1])[y]
    ece = 0.
    for low in np.arange(0, 1, .1):
        mask = (confidence >= low) & (confidence < low + .1 if low < .9 else confidence <= 1)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return {"rows": int(len(y)), "accuracy": float(correct.mean()),
            "nll": float(-np.log(np.clip(p[np.arange(len(y)), y], EPS, 1)).mean()),
            "brier": float(np.sum((p - onehot) ** 2, axis=1).mean()), "ece10": float(ece)}


def _write_csv(path, rows):
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def fit_meta(logits, y, repetitions, shifts, output: Path, metadata=None):
    """Fit and save a JSON-serializable meta model; never supply test rows.

    1/3/4: final temperatures, alpha, reliability heads. Head training inputs use
    leave-one-repetition crossfitted temperatures. 6: monotone scalar reliability
    calibration only. Final temperature is frozen before looking at rep 6.
    """
    z, s, yy, rr = _inputs(logits, shifts, y, repetitions)
    fit = np.isin(rr, FIT_REPS)
    cal = rr == CAL_REP
    temperatures, tdiag = fit_temperatures(z[fit], yy[fit], rr[fit])
    p_final = _probabilities(z, temperatures)
    p_crossfit = np.zeros_like(z[fit])
    crossfits = []
    for held in FIT_REPS:
        inner_train = fit & (rr != held)
        temp, opt = fit_temperatures(z[inner_train], yy[inner_train], rr[inner_train])
        p_crossfit[rr[fit] == held] = _probabilities(z[fit & (rr == held)], temp)
        crossfits.append({"held_repetition": held, "fit_repetitions": sorted(set(rr[inner_train].tolist())),
                          "temperatures": temp, "optimization": opt})
    q_fit = indicators(p_crossfit, s[fit])
    q_cal = indicators(p_final[cal], s[cal])
    # Positive temperature cannot change an expert's class argmax.
    correctness_fit = (z[fit].argmax(2) == yy[fit, None]).astype(float)
    correctness_cal = (z[cal].argmax(2) == yy[cal, None]).astype(float)
    weights_fit = _group_weights(rr[fit])
    alpha, adiag = _fit_alpha(p_crossfit, yy[fit], weights_fit)
    model = {"version": VERSION, "expert_order": list(EXPERTS), "num_classes": 17,
             "temperatures": temperatures, "alpha": alpha, "heads": {}, "reliability_calibrations": {},
             "metadata": metadata or {}, "provenance": {
                 "meta_fit_repetitions": list(FIT_REPS), "reliability_calibration_repetition": CAL_REP,
                 "outer_test_repetitions_forbidden_at_fit": [2, 5], "fit_rows": int(fit.sum()), "calibration_rows": int(cal.sum()),
                 "counts_per_repetition": {str(r): int((rr == r).sum()) for r in sorted(set(rr.tolist()))},
                 "temperatures_crossfit": crossfits, "final_temperature_optimization": tdiag, "alpha_optimization": adiag,
                 "alpha_fit_input": "crossfitted-temperature probabilities from repetitions 1/3/4",
                 "fit_digest": hashlib.sha256(z[fit].tobytes() + yy[fit].tobytes() + s[fit].tobytes() + rr[fit].tobytes()).hexdigest(),
                 "calibration_digest": hashlib.sha256(z[cal].tobytes() + yy[cal].tobytes() + s[cal].tobytes()).hexdigest(),
                 "fixed_hyperparameters": {"head_l2": HEAD_L2, "calibration_l2": CAL_L2, "alpha_l2": ALPHA_L2, "min_events": MIN_EVENTS},
                 "rule_inputs": ["normalized_entropy", "mean_pairwise_total_variation", "training_reference_deviation"],
                 "caveat": "internal development; shared base-model training histories mean this is not independent meta CV"}}
    support_rows, rule_rows, reliability_rows, initial_rows = [], [], [], []
    for name in ("logistic", "sugeno", "brb", "brb_no_shift"):
        model["heads"][name], model["reliability_calibrations"][name] = [], []
        for j, expert in enumerate(EXPERTS):
            head = _fit_head(name, q_fit[:, j], correctness_fit[:, j], weights_fit)
            raw_cal = _raw_head(head, q_cal[:, j])
            calibration = _fit_calibration(raw_cal, correctness_cal[:, j])
            model["heads"][name].append(head)
            model["reliability_calibrations"][name].append(calibration)
            for stage, values in [("before", raw_cal), ("after", _calibrated(raw_cal, calibration))]:
                reliability_rows.append({"method": name, "expert": expert, "stage": stage, "repetition": 6,
                    "scope": "calibration fitting rows; not independent evaluation", "rows": len(raw_cal),
                    "observed_correctness": float(correctness_cal[:, j].mean()), "mean_reliability": float(values.mean()),
                    "brier_binary": float(np.mean((values - correctness_cal[:, j]) ** 2)),
                    "nll_binary": _binary_loss(correctness_cal[:, j], values, np.full(len(values), 1 / len(values)))})
            if name != "logistic":
                qq = q_fit[:, j].copy()
                if name == "brb_no_shift":
                    qq[:, 2] = .5
                a = rule_activations(qq)
                beliefs = np.full(8, head["value"]) if head["kind"] == "constant" else expit(head["parameters"])
                for rule, bits in enumerate(RULE_BITS):
                    initial = float(head.get("prior", head.get("value")))
                    initial_rows.append({"method": name, "expert": expert, "rule": rule,
                        "uncertainty": int(bits[0]), "disagreement": int(bits[1]), "deviation": int(bits[2]),
                        "belief_incorrect": 1-initial, "belief_correct": initial,
                        "initial_logit": float(logit(initial)), "rule_weight": 1.0,
                        "antecedent_weights": "1,1,1", "reference_values": "0,1"})
                    rule_rows.append({"method": name, "expert": expert, "rule": rule,
                        "uncertainty": int(bits[0]), "disagreement": int(bits[1]), "deviation": int(bits[2]),
                        "belief_incorrect": float(1 - beliefs[rule]), "belief_correct": float(beliefs[rule])})
                    for rep in FIT_REPS:
                        mask = rr[fit] == rep
                        mass = a[mask, rule]
                        support_rows.append({"method": name, "expert": expert, "rule": rule, "repetition": rep,
                            "window_count": int(mask.sum()), "activation_mass": float(mass.sum()),
                            "mean_activation": float(mass.mean()), "windows_activation_above_0_1": int((mass > .1).sum()),
                            "effective_windows_kish_correlated_not_trials": float(mass.sum() ** 2 / max(np.sum(mass ** 2), EPS)),
                            "weighted_correctness": float(mass @ correctness_fit[mask, j] / max(mass.sum(), EPS))})
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    (output / "meta_model.json").write_text(json.dumps(model, indent=2, allow_nan=False), encoding="utf-8")
    (output / "meta_provenance.json").write_text(json.dumps(model["provenance"], indent=2, allow_nan=False), encoding="utf-8")
    _write_csv(output / "meta_rule_initial.csv", initial_rows)
    _write_csv(output / "meta_rule_conclusions.csv", rule_rows)
    _write_csv(output / "meta_rule_support_per_repetition.csv", support_rows)
    _write_csv(output / "meta_reliability_calibration.csv", reliability_rows)
    pred = predict_meta(model, z, s)
    metrics = [{"method": name, "repetition": int(rep),
                "scope": "meta-head development fit" if rep in FIT_REPS else "reliability calibration fit",
                **_metrics(yy[rr == rep], pp[rr == rep])}
               for name, pp in pred.items() for rep in sorted(set(rr.tolist()))]
    _write_csv(output / "meta_development_metrics.csv", metrics)
    return model


## taskaware_core.py
Label-free inference, exact neutral initialization, bounded corrections, two loss choices and analytical gradients. Rules are fusion-support scores, not calibrated probabilities.

In [ ]:
%%writefile /kaggle/working/db7_021_source/taskaware_core.py
"""Bounded, task-aware corrections to a frozen W/S/I probability mixture.

The caller supplies development predictions only to ``fit_adjustment``. This
numerical module does not choose hyperparameters, split rows, or access a test
set. ``row_weights`` can give each development repetition equal total weight.

Two small heads change the waveform and spectral log weights. The inertial log
weight stays fixed. For either head, a score in [-1, 1] produces a correction
in [-cap, cap], followed by normalization across all three experts. Thus an EMG
expert's weight ratio relative to inertial changes by at most exp(cap).

BRB conclusions here mean support for increasing versus decreasing the expert
weight, NOT a calibrated probability that the expert is correct. Analytical ER
combines eight complete binary rule conclusions using the frozen BRB engine.
"""
from __future__ import annotations

import numpy as np
from scipy.optimize import minimize
from scipy.special import expit, softmax

import brb_meta as b

VERSION = "db7-taskaware-adjustment-v1"
EXPERTS = ("W", "S", "I")
ANTECEDENTS = ("entropy", "disagreement", "training_deviation")
PARAMETER_L2 = 1e-4
PARAMETER_BOUND = 8.0
MAXITER = 250
LOSS_FLOOR = 1e-300


def _validate(kind, p, w0, q, cap, regularization=0.0, y=None, row_weights=None):
    """Validate probability geometry without silently altering the baseline."""
    if kind not in ("brb", "logistic"):
        raise ValueError("kind must be 'brb' or 'logistic'")
    p = np.asarray(p, dtype=np.float64)
    w0 = np.asarray(w0, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    if p.ndim != 3 or p.shape[1:] != (3, 17) or len(p) == 0:
        raise ValueError("p must be nonempty [N,3,17]")
    if w0.shape != p.shape[:2] or q.shape != (len(p), 3, 3):
        raise ValueError("w0 must be [N,3] and q must be [N,3,3]")
    if any(not np.isfinite(x).all() for x in (p, w0, q)):
        raise ValueError("p, w0, and q must be finite")
    if np.any((p < 0) | (p > 1)) or not np.allclose(p.sum(-1), 1, atol=1e-7, rtol=0):
        raise ValueError("each expert distribution must be normalized")
    if np.any((w0 < 0) | (w0 > 1)) or not np.allclose(w0.sum(-1), 1, atol=1e-7, rtol=0):
        raise ValueError("baseline weights must be nonnegative and normalized")
    if np.any((q < 0) | (q > 1)):
        raise ValueError("antecedents must lie in [0,1]")
    if not np.isfinite(cap) or cap < 0 or not np.isfinite(regularization) or regularization < 0:
        raise ValueError("cap and regularization must be finite and nonnegative")
    yy, rw = None, None
    if y is not None:
        yy = np.asarray(y)
        if yy.shape != (len(p),) or not np.isin(yy, np.arange(17)).all():
            raise ValueError("y must contain zero-based labels 0..16 with shape [N]")
        yy = yy.astype(np.int64)
        rw = np.asarray(row_weights, dtype=np.float64)
        if rw.shape != yy.shape or not np.isfinite(rw).all() or np.any(rw < 0) or rw.sum() <= 0:
            raise ValueError("row_weights must be finite nonnegative [N] with positive total")
        rw = rw / rw.sum()
    return p, w0, q, yy, rw


def _design(kind, q):
    if kind == "brb":
        return [b.rule_activations(q[:, j]) for j in range(2)]
    return [np.column_stack([np.ones(len(q)), q[:, j] - .5]) for j in range(2)]


def _forward(theta, kind, p, w0, design, cap, jacobian=False):
    k = 8 if kind == "brb" else 4
    theta = np.asarray(theta, dtype=np.float64)
    if theta.size != 2 * k or not np.isfinite(theta).all():
        raise ValueError(f"{kind} parameters must contain {2 * k} finite values")
    theta = theta.reshape(2, k)
    delta = np.zeros_like(w0)
    head_jacobians = []
    support_scores = np.zeros((len(p), 2))
    for j in range(2):
        if kind == "brb":
            beliefs = expit(theta[j])
            if jacobian:
                support, derivative = b.rimer_correct(design[j], beliefs, return_jacobian=True)
                derivative = derivative * (beliefs * (1 - beliefs))[None, :]
            else:
                support = b.rimer_correct(design[j], beliefs)
        else:
            support = expit(design[j] @ theta[j])
            if jacobian:
                derivative = design[j] * (support * (1 - support))[:, None]
        delta[:, j] = cap * (2 * support - 1)
        support_scores[:, j] = support
        if jacobian:
            head_jacobians.append(derivative)
    # Exact zero baseline weights remain zero; there is always positive mass in
    # at least one expert, so softmax is well defined even with log(0)=-inf.
    with np.errstate(divide="ignore"):
        weights = softmax(np.log(w0) + delta, axis=1)
    probabilities = np.sum(weights[..., None] * p, axis=1)
    return probabilities, weights, delta, head_jacobians, support_scores


def _objective_prepared(theta, kind, p, w0, design, y, row_weights, cap, regularization, loss_kind):
    probabilities, weights, _, jacobians, scores = _forward(theta, kind, p, w0, design, cap, True)
    true_expert_p = p[np.arange(len(p))[:, None], np.arange(3)[None, :], y[:, None]]
    true_mixture_p = probabilities[np.arange(len(p)), y]
    safe_p = np.maximum(true_mixture_p, LOSS_FLOOR)
    if loss_kind == "fusion_ce":
        cross_entropy = -np.sum(row_weights * np.log(safe_p))
    else:
        target = (p[:, :2].argmax(2) == y[:, None]).astype(float)
        safe_scores = np.clip(scores, 1e-12, 1 - 1e-12)
        cross_entropy = -.5 * np.sum(row_weights[:, None] * (
            target * np.log(safe_scores) + (1 - target) * np.log1p(-safe_scores)))
    difference = weights - w0
    displacement = np.sum(row_weights * np.sum(difference ** 2, axis=1))
    theta = np.asarray(theta, dtype=np.float64).reshape(-1)
    loss = cross_entropy + regularization * displacement + PARAMETER_L2 * np.mean(theta ** 2)

    # CE derivative through the softmax log-weight correction. Multiplying by
    # weights before division avoids enormous 1/p intermediate values.
    derivative_delta = 2 * regularization * row_weights[:, None] * weights * (
        difference - np.sum(weights * difference, axis=1, keepdims=True))
    if loss_kind == "fusion_ce":
        ce_delta = row_weights[:, None] * (weights - weights * true_expert_p / safe_p[:, None])
        ce_delta[true_mixture_p <= LOSS_FLOOR] = 0.0
        derivative_delta += ce_delta
        gradient = np.concatenate([2 * cap * derivative_delta[:, j] @ jacobians[j] for j in range(2)])
    else:
        derivative_score = .5 * row_weights[:, None] * (safe_scores - target) / (safe_scores * (1 - safe_scores))
        derivative_score[(scores <= 1e-12) | (scores >= 1 - 1e-12)] = 0.0
        gradient = np.concatenate([
            (derivative_score[:, j] + 2 * cap * derivative_delta[:, j]) @ jacobians[j]
            for j in range(2)])
    gradient += 2 * PARAMETER_L2 * theta / len(theta)
    return float(loss), gradient


def objective(theta, kind, p, w0, q, y, row_weights, cap, regularization, loss="fusion_ce"):
    """Return the final-fusion objective and its analytic parameter gradient.

    The objective is weighted gesture cross-entropy, plus ``regularization``
    times weighted squared displacement from baseline weights, plus 1e-4 mean
    squared head parameters. This function is public for finite-difference
    checks. With ``loss='expert_bce'``, replace gesture CE with the mean binary
    expert-correctness loss for the W/S scores; mapping and penalties stay the
    same. Fitting caches the head design matrices for efficiency.
    """
    p, w0, q, y, rw = _validate(kind, p, w0, q, cap, regularization, y, row_weights)
    if loss not in ("fusion_ce", "expert_bce"):
        raise ValueError("loss must be 'fusion_ce' or 'expert_bce'")
    return _objective_prepared(theta, kind, p, w0, _design(kind, q), y, rw, cap, regularization, loss)


def fit_adjustment(kind, p, w0, q, y, row_weights, cap, regularization, loss="fusion_ce"):
    """Fit two correction heads on caller-provided development rows only.

    Returns a JSON-serializable model. The neutral all-zero parameters exactly
    reproduce the baseline mixture. A nonfinite or worse candidate is discarded
    in favor of this neutral model. An improved finite but unconverged candidate
    is retained with ``optimization.success=False`` and an explicit status;
    callers must report that status rather than claiming successful convergence.
    A zero cap is the prespecified neutral control: no optimizer is invoked and
    all head parameters remain zero, including for the expert-BCE objective.
    """
    p, w0, q, y, rw = _validate(kind, p, w0, q, cap, regularization, y, row_weights)
    if loss not in ("fusion_ce", "expert_bce"):
        raise ValueError("loss must be 'fusion_ce' or 'expert_bce'")
    design = _design(kind, q)
    theta0 = np.zeros(16 if kind == "brb" else 8)
    args = (kind, p, w0, design, y, rw, cap, regularization, loss)
    initial_loss, _ = _objective_prepared(theta0, *args)
    chosen = theta0.copy()
    diagnostic = {"success": False, "accepted_candidate": False,
                  "initial_objective": float(initial_loss), "neutral_fallback": False}
    if cap == 0:
        diagnostic.update({"success": True, "message": "Zero cap: prespecified neutral control; optimizer skipped",
                           "iterations": 0, "evaluations": 0, "candidate_objective": None,
                           "status": "neutral_prespecified", "fallback_reason": None})
    else:
        try:
            opt = minimize(_objective_prepared, theta0, args=args, jac=True, method="L-BFGS-B",
                           bounds=[(-PARAMETER_BOUND, PARAMETER_BOUND)] * len(theta0),
                           options={"maxiter": MAXITER, "ftol": 1e-11, "gtol": 1e-7, "maxls": 40})
            finite = bool(np.isfinite(opt.fun) and np.isfinite(opt.x).all())
            candidate_loss = float(opt.fun) if finite else None
            acceptable = finite and opt.fun <= initial_loss + 1e-12
            if acceptable:
                chosen = np.asarray(opt.x, dtype=np.float64)
            diagnostic.update({"success": bool(opt.success), "message": str(opt.message),
                               "iterations": int(opt.nit), "evaluations": int(opt.nfev),
                               "candidate_objective": candidate_loss,
                               "accepted_candidate": bool(acceptable),
                               "neutral_fallback": not acceptable,
                               "status": ("converged" if opt.success else "unconverged_candidate")
                                         if acceptable else "neutral_fallback",
                               "fallback_reason": None if acceptable else
                               ("nonfinite candidate" if not finite else "candidate worsened objective")})
        except (FloatingPointError, ValueError, RuntimeError) as exc:
            diagnostic.update({"message": str(exc), "iterations": 0, "evaluations": 0,
                               "candidate_objective": None, "neutral_fallback": True,
                               "status": "neutral_fallback", "fallback_reason": "optimizer exception"})
    final_loss, final_gradient = _objective_prepared(chosen, *args)
    diagnostic.update({"selected_objective": float(final_loss),
                       "selected_gradient_infinity_norm": float(np.max(np.abs(final_gradient)))})
    return {"version": VERSION, "kind": kind,
            "parameters": chosen.reshape(2, -1).tolist(),
            "initial_parameters": theta0.reshape(2, -1).tolist(),
            "expert_order": list(EXPERTS), "adjusted_experts": ["W", "S"],
            "inertial_log_weight_adjustment": 0.0,
            "cap": float(cap), "regularization": float(regularization),
            "parameter_l2": PARAMETER_L2, "parameter_bounds": [-PARAMETER_BOUND, PARAMETER_BOUND],
            "training_rows": int(len(p)), "optimization": diagnostic,
            "brb_consequent_semantics": "support for increasing versus decreasing expert log weight",
            "score_semantics": "fusion-support scores; not calibrated correctness probabilities",
            "loss": loss,
            "training_target": "final gesture label" if loss == "fusion_ce" else "W/S expert argmax correctness",
            "objective_description": ("weighted gesture cross-entropy" if loss == "fusion_ce" else
                                      "mean W/S expert binary correctness cross-entropy") +
                                     " + lambda weighted squared weight displacement + 1e-4 mean parameter square"}


def predict_adjustment(model, p, w0, q):
    """Return ``probabilities[N,17]``, ``weights[N,3]``, and ``delta[N,3]``.

    No labels are needed or accepted. ``delta`` is the pre-normalization log
    weight correction, and the inertial column is identically zero.
    """
    if model.get("version") != VERSION:
        raise ValueError("unsupported adjustment model version")
    kind, cap = model["kind"], float(model["cap"])
    p, w0, q, _, _ = _validate(kind, p, w0, q, cap)
    probabilities, weights, delta, _, scores = _forward(
        model["parameters"], kind, p, w0, _design(kind, q), cap)
    return {"probabilities": probabilities, "weights": weights, "delta": delta,
            "support_scores": scores}


def rule_export_rows(model):
    """Readable initial/trained BRB conclusions, or logistic coefficients.

    Reference values remain 0 (low) and 1 (high). Rule/attribute weights are
    fixed at one, with product matching followed by normalized ER aggregation.
    Only the eight conclusion logits per EMG head are trained for BRB.
    """
    rows = []
    for stage, key in (("initial", "initial_parameters"), ("trained", "parameters")):
        params = np.asarray(model[key], dtype=float)
        for j, expert in enumerate(EXPERTS[:2]):
            if model["kind"] == "brb":
                for k, bits in enumerate(b.RULE_BITS):
                    support = float(expit(params[j, k]))
                    rows.append({"stage": stage, "expert": expert, "rule": k + 1,
                                 **{f"{name}_reference": int(bit) for name, bit in zip(ANTECEDENTS, bits)},
                                 "rule_weight": 1.0, "logit": float(params[j, k]),
                                 "belief_increase": support, "belief_decrease": 1 - support,
                                 "rule_vertex_delta": float(model["cap"] * (2 * support - 1))})
            else:
                for k, name in enumerate(("intercept", *ANTECEDENTS)):
                    rows.append({"stage": stage, "expert": expert, "coefficient": name,
                                 "value": float(params[j, k])})
    return rows


## taskaware_study.py
Separate training-side development selection from test evaluation. This code locks the selected bounds and penalties before reading test arrays, then exports every prespecified arm.

In [ ]:
%%writefile /kaggle/working/db7_021_source/taskaware_study.py
"""DB7-021: training-side selected, bounded task-aware BRB corrections."""
from pathlib import Path
import io,json,hashlib,zipfile,shutil
import numpy as np
import pandas as pd
from scipy.stats import binomtest
import brb_meta as b
from taskaware_core import fit_adjustment,predict_adjustment,rule_export_rows

SOURCE_SHA='65cf7bf7b914e0eb2f4178e96cffc74d27b6d42dacd694a6eaac42c4aec54eb0'
ARMS={'task_brb':('brb','fusion_ce'),'task_logistic':('logistic','fusion_ce'),'correctness_brb':('brb','expert_bce')}
GRID=[{'id':'neutral','cap':0.,'regularization':1.}]+[{'id':f'ratio{ratio}_l{lam}','cap':float(np.log(ratio)),'regularization':lam} for ratio in [1.25,2.] for lam in [.1,1.]]

def dump(path,value):
    path.parent.mkdir(parents=True,exist_ok=True);path.write_text(json.dumps(value,indent=2,allow_nan=False),encoding='utf-8')

def fit_development_baseline(z,y,r,s,fit_reps):
    """Caller supplies only two fit repetitions and calibration rep6; held labels absent."""
    assert set(r.tolist())==set(fit_reps)|{6} and len(fit_reps)==2 and set(fit_reps).issubset({1,3,4})
    fit=np.isin(r,fit_reps);cal=r==6
    temp,tdiag=b.fit_temperatures(z[fit],y[fit],r[fit]);p=b._probabilities(z,temp)
    cf=np.zeros_like(z[fit]);cross=[]
    for held in fit_reps:
        train=fit&(r!=held);tt,diag=b.fit_temperatures(z[train],y[train],r[train]);cf[r[fit]==held]=b._probabilities(z[fit&(r==held)],tt)
        cross.append(dict(held_repetition=int(held),fit_repetitions=[int(v) for v in fit_reps if v!=held],temperatures=tt,optimization=diag))
    q=b.indicators(cf,s[fit]);qc=b.indicators(p[cal],s[cal]);weights=b._group_weights(r[fit]);alpha,adiag=b._fit_alpha(cf,y[fit],weights)
    heads=[];cals=[]
    for j in range(3):
        head=b._fit_head('brb',q[:,j],(z[fit,j].argmax(1)==y[fit]).astype(float),weights)
        cals.append(b._fit_calibration(b._raw_head(head,qc[:,j]),(z[cal,j].argmax(1)==y[cal]).astype(float)));heads.append(head)
    model=dict(version=b.VERSION,temperatures=temp,alpha=alpha,heads={'brb':heads},reliability_calibrations={'brb':cals},provenance=dict(fit_repetitions=list(fit_reps),calibration_repetition=6,crossfit=cross,temperature_optimization=tdiag,alpha_optimization=adiag))
    confcal=b._fit_calibration(p[cal,2].max(1),(z[cal,2].argmax(1)==y[cal]).astype(float))
    return model,confcal

def final_confidence_calibration(model,oof):
    cal=oof['oof_repetitions']==6
    p=b._probabilities(oof['oof_logits'][cal],model['temperatures'])
    return b._fit_calibration(p[:,2].max(1),(p[:,2].argmax(1)==oof['oof_y'][cal]).astype(float))

def baseline_arrays(model,confcal,z,s):
    d=b.predict_diagnostics(model,z,s);p=d['expert_probabilities'];rr=d['reliabilities']['brb'].copy()
    rr[:,2]=b._calibrated(p[:,2].max(1),confcal)
    w=b._weights(model['alpha'],rr)
    return p,w,d['indicators']

def prediction_tuple(model,p,w,q):
    d=predict_adjustment(model,p,w,q)
    return d['probabilities'],d['weights'],d['delta']

def train_candidate(arm,candidate,arrays,y,groups):
    kind,loss=ARMS[arm];p,w,q=arrays
    return fit_adjustment(kind,p,w,q,y,b._group_weights(groups),candidate['cap'],candidate['regularization'],loss=loss)

def development_subject(subject,oof,out):
    """No test keys accepted. Three held-meta repetition scores, not independent neural CV."""
    assert set(oof)=={'oof_logits','oof_y','oof_repetitions','oof_shifts'}
    z=oof['oof_logits'];y=oof['oof_y'];r=oof['oof_repetitions'];s=oof['oof_shifts'];assert set(r.tolist())=={1,3,4,6}
    rows=[]
    for held in [1,3,4]:
        fit_reps=[v for v in [1,3,4] if v!=held];base_mask=np.isin(r,fit_reps+[6]);fit=np.isin(r,fit_reps);val=r==held
        model,cal=fit_development_baseline(z[base_mask],y[base_mask],r[base_mask],s[base_mask],fit_reps)
        folder=out/f'development/S{subject:02d}/held{held}';dump(folder/'baseline.json',dict(model=model,confidence_calibration=cal))
        ta=baseline_arrays(model,cal,z[fit],s[fit]);va=baseline_arrays(model,cal,z[val],s[val]);bp=(va[0]*va[1][:,:,None]).sum(1)
        bm=b._metrics(y[val],bp)
        for arm in ARMS:
            for candidate in GRID:
                fitted=train_candidate(arm,candidate,ta,y[fit],r[fit]);pred,_,_=prediction_tuple(fitted,*va)
                dump(folder/(arm+'_'+candidate['id']+'.json'),fitted)
                rows.append(dict(subject=subject,held_repetition=held,arm=arm,candidate=candidate['id'],cap=candidate['cap'],regularization=candidate['regularization'],baseline_accuracy=bm['accuracy'],baseline_nll=bm['nll'],**b._metrics(y[val],pred)))
        print('DEVELOPMENT',subject,'held',held,flush=True)
    return rows

def select_candidates(rows,out):
    df=pd.DataFrame(rows);df.to_csv(out/'development_metrics.csv',index=False)
    # One row per subject×held repetition, averaged equally. Every arm has same search budget.
    grouped=df.groupby(['arm','candidate','cap','regularization'],as_index=False)[['accuracy','nll','baseline_accuracy','baseline_nll']].mean()
    grouped.to_csv(out/'development_selection_table.csv',index=False);selected={}
    for arm in ARMS:
        a=grouped[grouped.arm==arm];ok=a[(a.accuracy>=a.baseline_accuracy-1e-12)&(a.nll<a.baseline_nll-1e-12)]
        if len(ok):best=ok.sort_values(['nll','cap','regularization'],ascending=[True,True,False]).iloc[0];cid=best.candidate
        else:cid='neutral'
        selected[arm]=next(c for c in GRID if c['id']==cid)
    record=dict(selected=selected,selection_uses_test=False,criterion='lowest mean held-meta NLL subject to mean accuracy >= baseline; ties smaller cap then larger penalty; no qualifying improvement -> neutral',scope='exploratory development: base neural OOF models share training histories',grid=GRID)
    dump(out/'SELECTION_LOCKED_BEFORE_TEST.json',record)
    return selected

def evaluate_subject(subject,bundle,model,selected,out):
    oof={k:bundle[k] for k in ['oof_logits','oof_y','oof_repetitions','oof_shifts']};fit=np.isin(oof['oof_repetitions'],[1,3,4]);cal=final_confidence_calibration(model,oof)
    train_arrays=baseline_arrays(model,cal,oof['oof_logits'][fit],oof['oof_shifts'][fit]);folder=out/f'final/S{subject:02d}'
    fitted={arm:train_candidate(arm,selected[arm],train_arrays,oof['oof_y'][fit],oof['oof_repetitions'][fit]) for arm in ARMS}
    dump(folder/'fitted_adjustments.json',dict(confidence_calibration=cal,models=fitted))
    for arm,head in fitted.items():pd.DataFrame(rule_export_rows(head)).to_csv(folder/(arm+'_initial_trained_rules.csv'),index=False)
    # Model fitting and global selection have finished before test arrays are used below.
    p,w,q=baseline_arrays(model,cal,bundle['test_logits'],bundle['test_shifts']);y=bundle['test_y']
    probs=b.predict_meta(model,bundle['test_logits'],bundle['test_shifts']);probs['hybrid']=(p*w[:,:,None]).sum(1)
    probs.update(control_si=bundle['control_si'],control_wsi=bundle['control_wsi'])
    trace={'label':y+1};params=[]
    for arm,head in fitted.items():
        pp,ww,dd=prediction_tuple(head,p,w,q);probs[arm]=pp
        for j,branch in enumerate('WSI'):
            trace[arm+'_'+branch+'_weight']=ww[:,j];trace[arm+'_'+branch+'_delta']=dd[:,j]
        # Detailed initial/trained parameter dumps remain in fitted_adjustments.json.
    rows=[];pairs=[]
    report_arms=['hybrid','brb','confidence_weight','global_weight','expert_i','control_si','control_wsi',*ARMS]
    for arm in report_arms:
        pp=probs[arm];assert np.isfinite(pp).all() and np.allclose(pp.sum(1),1,atol=1e-6)
        predicted=pp.argmax(1);correct=predicted==y;trace[arm+'_pred']=predicted+1;trace[arm+'_correct']=correct
        conf=np.zeros((17,17),int);np.add.at(conf,(y,predicted),1);pd.DataFrame(conf,index=range(1,18),columns=range(1,18)).to_csv(folder/(arm+'_confusion.csv'))
        recall=np.diag(conf)/np.maximum(conf.sum(1),1);f1=2*np.diag(conf)/np.maximum(conf.sum(1)+conf.sum(0),1)
        disagree=(p.argmax(2)!=p.argmax(2)[:,[0]]).any(1)
        rows.append(dict(subject=subject,arm=arm,macro_f1=float(f1.mean()),balanced_accuracy=float(recall.mean()),disagreement_accuracy=float(correct[disagree].mean()) if disagree.any() else None,**b._metrics(y,pp)))
        for ref in ['hybrid','brb','control_si','task_logistic','correctness_brb']:
            rc=probs[ref].argmax(1)==y;rec=int((correct&~rc).sum());harm=int((~correct&rc).sum())
            pairs.append(dict(subject=subject,arm=arm,reference=ref,recovered=rec,harmed=harm,net=rec-harm,mcnemar_window_p_descriptive=binomtest(rec,rec+harm,.5).pvalue if rec+harm else 1.))
    return rows,pairs,pd.DataFrame(trace),fitted

def summarize(rows,pairs,out):
    df=pd.DataFrame(rows);df.to_csv(out/'subject_metrics.csv',index=False);pd.DataFrame(pairs).to_csv(out/'recovery_harm.csv',index=False)
    means=df.groupby('arm')[['accuracy','macro_f1','balanced_accuracy','nll','brier','ece10']].mean();means.to_csv(out/'mean_subject_metrics.csv')
    pivot=df.pivot(index='subject',columns='arm',values='accuracy');rng=np.random.default_rng(42021);tests=[]
    contrasts=[('task_brb','hybrid'),('task_logistic','hybrid'),('task_brb','task_logistic'),('task_brb','correctness_brb')]
    for arm,ref in contrasts:
        delta=(pivot[arm]-pivot[ref]).to_numpy();n=len(delta);count=0
        for _ in range(100):count+=int((np.abs((rng.choice([-1,1],(1000,n))*delta).mean(1))>=abs(delta.mean())-1e-15).sum())
        boots=delta[rng.integers(n,size=(20000,n))].mean(1)
        tests.append(dict(arm=arm,reference=ref,mean_change_pp=100*delta.mean(),ci_low_pp=100*np.quantile(boots,.025),ci_high_pp=100*np.quantile(boots,.975),p=(count+1)/100001))
    running=0
    for rank,index in enumerate(np.argsort([x['p'] for x in tests])):running=max(running,min(1,tests[index]['p']*(len(tests)-rank)));tests[index]['p_holm']=running
    pd.DataFrame(tests).to_csv(out/'subject_significance.csv',index=False)
    optrows=[]
    for file in out.glob('development/S*/held*/*.json'):
        a=json.loads(file.read_text())
        if 'optimization' in a:optrows.append(dict(scope='development',path=str(file.relative_to(out)),**a['optimization']))
    for file in out.glob('final/S*/fitted_adjustments.json'):
        for arm,a in json.loads(file.read_text())['models'].items():optrows.append(dict(scope='final',path=str(file.relative_to(out))+':'+arm,**a['optimization']))
    pd.DataFrame(optrows).to_csv(out/'optimization_status.csv',index=False)
    import matplotlib;matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    ax=pivot[['hybrid',*ARMS]].mul(100).plot(figsize=(12,5),marker='o');ax.set_ylabel('Accuracy (%)');ax.figure.tight_layout();ax.figure.savefig(out/'subject_accuracy.png',dpi=150);plt.close(ax.figure)
    report='''# DB7-021 task-aware bounded BRB comparison
Neural experts frozen from DB7-018; all22 subjects, seed42,200ms/10ms,train1/3/4/6,test2/5.
W/S residual BRB or logistic scores adjust weights around the DB7-020 calibrated-confidence hybrid.
Inertial log-weight adjustment is zero; normalization may still change its final effective weight.
Task-aware scores are fusion-support scores, not calibrated correctness probabilities.
The matched correctness-BRB control uses identical architecture with an expert-correctness objective.
All methods receive identical development search budgets; selected settings may differ and are reported.

Development: held meta repetition1/3/4 in turn; other2 fit reference meta models and adjustments;
OOF6 calibrates reference reliability. Global settings are locked before test evaluation.
Saved neural OOF models share training histories: these are exploratory development folds,
not independent nested validation. Tests were previously inspected, and neural seed remains42.
Bounds constrain log-weight changes relative to inertial, not absolute percentage-point weight changes.
No improvement or reduced harm is guaranteed. A neutral fallback can be selected.
Subject paired sign-flip tests, Holm across4 primary comparisons; bootstrap intervals unadjusted.
Window McNemar is descriptive only because95% overlapping windows are correlated.

Mean subject metrics:
'''
    (out/'REPORT.md').write_text(report+'\n```\n'+means.to_string()+'\n```\n',encoding='utf-8')

def main():
    out=Path('/kaggle/working/db7_021_taskaware');out.mkdir(exist_ok=True)
    sources=list(Path('/kaggle/input').rglob('db7_brb_full_20260915T013327619764Z.zip'));assert len(sources)==1
    h=hashlib.sha256()
    with sources[0].open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    assert h.hexdigest()==SOURCE_SHA
    dev=[]
    with zipfile.ZipFile(sources[0]) as z:
        done=json.loads(z.read('completion.json'));assert done['success'] and done['neural_fits']==374 and done['meta_completed']==22
        names=sorted(n for n in z.namelist() if n.startswith('full/') and n.endswith('/subject_bundle.npz'));assert len(names)==22
        # First pass materializes only OOF members of each NPZ. Test arrays are not loaded.
        for name in names:
            base=name.rsplit('/',1)[0];model=json.loads(z.read(base+'/meta/meta_model.json'));subject=int(model['metadata']['subject'])
            with np.load(io.BytesIO(z.read(name)),allow_pickle=False) as a:oof={k:a[k] for k in ['oof_logits','oof_y','oof_repetitions','oof_shifts']}
            dev.extend(development_subject(subject,oof,out))
        selected=select_candidates(dev,out);print('SELECTION LOCKED',json.dumps(selected),flush=True)
        rows=[];pairs=[];subjects=[]
        for name in names:
            base=name.rsplit('/',1)[0];model=json.loads(z.read(base+'/meta/meta_model.json'));subject=int(model['metadata']['subject'])
            with np.load(io.BytesIO(z.read(name)),allow_pickle=False) as a:bundle={k:a[k] for k in a.files}
            rr,pp,trace,models=evaluate_subject(subject,bundle,model,selected,out)
            original=pd.read_csv(io.BytesIO(z.read(base+'/results/method_metrics.csv'))).set_index('method')
            assert np.isclose(next(x['accuracy'] for x in rr if x['arm']=='brb'),original.loc['brb','accuracy'],atol=1e-12,rtol=0)
            meta=pd.read_csv(io.BytesIO(z.read(base+'/test_metadata.csv')));assert np.array_equal(meta.gesture,trace.label)
            trace=pd.concat([meta,trace],axis=1);folder=out/f'final/S{subject:02d}';trace.to_csv(folder/'test_trace.csv.gz',index=False,compression='gzip')
            errors=[]
            for (gesture,rep),g in trace.groupby(['gesture','native_repetition']):
                for arm in ['hybrid',*ARMS]:errors.append(dict(subject=subject,gesture=gesture,repetition=rep,arm=arm,windows=len(g),wrong=int((~g[arm+'_correct']).sum())))
            pd.DataFrame(errors).to_csv(folder/'gesture_repetition.csv',index=False)
            rows.extend(rr);pairs.extend(pp);subjects.append(subject);print('TEST COMPLETE',subject,flush=True)
    assert sorted(subjects)==list(range(1,23));summarize(rows,pairs,out)
    dump(out/'completion.json',dict(experiment_id='DB7-021',success=True,subjects=sorted(subjects),source_archive_sha256=SOURCE_SHA,neural_fits=0,test_used_for_fitting=False,selection_locked_before_test=True,development_folds=66,development_adjustment_candidates=990,final_adjustment_fits=66,selected=selected))
    shutil.make_archive(str(out),'zip',out);print('DB7-021 COMPLETE',flush=True)
if __name__=='__main__':main()


## Run once on the verified saved archive
No retraining fallback and no test-based selection.

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,str(SOURCE/'taskaware_study.py')],check=True)


In [ ]:
from IPython.display import display,Markdown,Image
import pandas as pd
OUT=Path('/kaggle/working/db7_021_taskaware')
display(Markdown((OUT/'REPORT.md').read_text()))
display(pd.read_csv(OUT/'development_selection_table.csv'))
display(pd.read_csv(OUT/'subject_significance.csv'))
display(Image(filename=str(OUT/'subject_accuracy.png')))
